# Big Cities Health Data Inventory — Data Cleaning

## 0. Import & Load Data

In [23]:
import pandas as pd

df = pd.read_csv('Big_Cities_Health_Data_Inventory.csv')

## 1. Exploratory Data Analysis

In [24]:
df.shape

(13509, 7)

In [25]:
df.head()

,Indicator Category,Indicator,Year,Gender,Race/ Ethnicity,Value,Place
0,HIV/AIDS,"AIDS Diagnoses Rate (Per 100,000 people)",2013,Both,All,30.4,"Atlanta (Fulton County), GA"
1,HIV/AIDS,"AIDS Diagnoses Rate (Per 100,000 people)",2012,Both,All,39.6,"Atlanta (Fulton County), GA"
2,HIV/AIDS,"AIDS Diagnoses Rate (Per 100,000 people)",2011,Both,All,41.7,"Atlanta (Fulton County), GA"
3,HIV/AIDS,"AIDS Diagnoses Rate (Per 100,000 people)",2011,Both,All,41.7,"Atlanta (Fulton County), GA"
4,Cancer,All Types of Cancer Mortality Rate (Age-Adjust...,2013,Male,All,195.8,"Atlanta (Fulton County), GA"


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13509 entries, 0 to 13508
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Indicator Category  13509 non-null  object 
 1   Indicator           13509 non-null  object 
 2   Year                13509 non-null  object 
 3   Gender              13509 non-null  object 
 4   Race/ Ethnicity     13509 non-null  object 
 5   Value               13496 non-null  float64
 6   Place               13509 non-null  object 
dtypes: float64(1), object(6)
memory usage: 738.9+ KB


In [27]:
df.describe()

,Value
count,13496.000000
mean,285.751052
std,3193.018958
min,0.000000
25%,7.000000
50%,16.100000
75%,45.225000
max,80977.000000


In [28]:
df.isnull().sum()

Indicator Category     0
Indicator              0
Year                   0
Gender                 0
Race/ Ethnicity        0
Value                 13
Place                  0
dtype: int64

In [29]:
print(f'Duplicate rows: {df.duplicated().sum()}')
df[df.duplicated(keep=False)]

Duplicate rows: 1


,Indicator Category,Indicator,Year,Gender,Race/ Ethnicity,Value,Place
2,HIV/AIDS,"AIDS Diagnoses Rate (Per 100,000 people)",2011,Both,All,41.7,"Atlanta (Fulton County), GA"
3,HIV/AIDS,"AIDS Diagnoses Rate (Per 100,000 people)",2011,Both,All,41.7,"Atlanta (Fulton County), GA"


In [30]:
df.nunique()

Indicator Category      11
Indicator               44
Year                    13
Gender                   3
Race/ Ethnicity          9
Value                 2677
Place                   29
dtype: int64

In [31]:
print(sorted(df['Year'].unique()))

['2003-2012', '2003-2013', '2004-2013', '2007-2012', '2008-2012', '2010', '2011', '2011-2012', '2011-2013', '2012', '2013', '2014', '2015']


In [32]:
for col in ['Indicator Category', 'Gender', 'Race/ Ethnicity']:
    print(f'{col}: {df[col].unique()}\n')

Indicator Category: ['HIV/AIDS' 'Cancer' 'Maternal and Child Health'
 'Life Expectancy and Death Rate (Overall)'
 'Nutrition, Physical Activity, & Obesity'
 'Behavioral Health/Substance Abuse' 'Injury and Violence' 'Demographics'
 'Infectious Disease' 'Tobacco' 'Food Safety']

Gender: ['Both' 'Male' 'Female']

Race/ Ethnicity: ['All' 'Black' 'White' 'Asian/PI' 'Hispanic' 'Multiracial' 'Other'
 'American Indian/Alaska Native' 'Native American']



In [33]:
df['Place'].unique()

array(['Atlanta (Fulton County), GA', 'Cleveland, OH', 'Baltimore, MD',
       'Boston, MA', 'Portland (Multnomah County), OR', 'Chicago, IL',
       'San Diego County, CA', 'Dallas, TX', 'Denver, CO', 'Detroit, MI',
       'Kansas City, MO', 'Fort Worth (Tarrant County), TX',
       'Houston, TX', 'Seattle, WA', 'Washington, DC', 'Los Angeles, CA',
       'Las Vegas (Clark County), NV', 'Miami (Miami-Dade County), FL',
       'San Jose, CA', 'Minneapolis, MN', 'New York, NY',
       'Philadelphia, PA', 'Oakland, CA', 'Phoenix, AZ', 'Sacramento, CA',
       'San Antonio, TX', 'San Francisco, CA', 'U.S. Total',
       'Long Beach, CA'], dtype=object)

## 2. Data Cleaning

In [34]:
# drop duplicate rows
df = df.drop_duplicates()

In [35]:
# standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.replace(r'[/\s]+', '_', regex=True)
)
print(df.columns.tolist())

['Indicator_Category', 'Indicator', 'Year', 'Gender', 'Race_Ethnicity', 'Value', 'Place']


In [36]:
# drop rows where Value is missing
df = df.dropna(subset=['Value'])

In [37]:
# split Year into Start_Year and End_Year
df['Start_Year'] = df['Year'].str.split('-').str[0].astype(int)
df['End_Year']   = df['Year'].str.split('-').str[-1].astype(int)

df[df['Year'].str.contains('-')][['Year', 'Start_Year', 'End_Year']].head()

,Year,Start_Year,End_Year
51,2003-2012,2003,2012
367,2011-2013,2011,2013
732,2008-2012,2008,2012
733,2008-2012,2008,2012
734,2008-2012,2008,2012


In [38]:
# extract State and City from Place
df['State'] = df['Place'].str.extract(r',\s*([A-Z]{2})$')
df['City']  = df['Place'].str.extract(r'^(.*?)(?:,|\()')

df[['Place', 'City', 'State']].drop_duplicates().head(10)

,Place,City,State
0,"Atlanta (Fulton County), GA",Atlanta,GA
121,"Cleveland, OH",Cleveland,OH
201,"Baltimore, MD",Baltimore,MD
542,"Boston, MA",Boston,MA
693,"Portland (Multnomah County), OR",Portland,OR
989,"Chicago, IL",Chicago,IL
1198,"San Diego County, CA",San Diego County,CA
1803,"Dallas, TX",Dallas,TX
1948,"Denver, CO",Denver,CO
2652,"Detroit, MI",Detroit,MI


In [39]:
# verify NaN only comes from 'U.S. Total'
print('State NaN:', df[df['State'].isnull()]['Place'].unique())
print('City  NaN:', df[df['City'].isnull()]['Place'].unique())

State NaN: ['U.S. Total']
City  NaN: ['U.S. Total']


## 3. Validation

In [40]:
df.shape

(13495, 11)

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13495 entries, 0 to 13508
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Indicator_Category  13495 non-null  object 
 1   Indicator           13495 non-null  object 
 2   Year                13495 non-null  object 
 3   Gender              13495 non-null  object 
 4   Race_Ethnicity      13495 non-null  object 
 5   Value               13495 non-null  float64
 6   Place               13495 non-null  object 
 7   Start_Year          13495 non-null  int64  
 8   End_Year            13495 non-null  int64  
 9   State               12884 non-null  object 
 10  City                12884 non-null  object 
dtypes: float64(1), int64(2), object(8)
memory usage: 1.2+ MB


In [42]:
df.isnull().sum()

Indicator_Category      0
Indicator               0
Year                    0
Gender                  0
Race_Ethnicity          0
Value                   0
Place                   0
Start_Year              0
End_Year                0
State                 611
City                  611
dtype: int64

In [43]:
df.head()

,Indicator_Category,Indicator,Year,Gender,Race_Ethnicity,Value,Place,Start_Year,End_Year,State,City
0,HIV/AIDS,"AIDS Diagnoses Rate (Per 100,000 people)",2013,Both,All,30.4,"Atlanta (Fulton County), GA",2013,2013,GA,Atlanta
1,HIV/AIDS,"AIDS Diagnoses Rate (Per 100,000 people)",2012,Both,All,39.6,"Atlanta (Fulton County), GA",2012,2012,GA,Atlanta
2,HIV/AIDS,"AIDS Diagnoses Rate (Per 100,000 people)",2011,Both,All,41.7,"Atlanta (Fulton County), GA",2011,2011,GA,Atlanta
4,Cancer,All Types of Cancer Mortality Rate (Age-Adjust...,2013,Male,All,195.8,"Atlanta (Fulton County), GA",2013,2013,GA,Atlanta
5,Cancer,All Types of Cancer Mortality Rate (Age-Adjust...,2013,Female,All,135.5,"Atlanta (Fulton County), GA",2013,2013,GA,Atlanta


## 4. Save

In [44]:
df.to_csv('Big_Cities_Health_Data_Inventory_cleaned.csv', index=False)
print('Saved: Big_Cities_Health_Data_Inventory_cleaned.csv')

Saved: Big_Cities_Health_Data_Inventory_cleaned.csv
